# Лабораторная работа №6

In [1]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "Helsinki-NLP/opus-mt-en-ru"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)


C:\Users\magofrays\PycharmProjects\lb6\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 64778.83it/s]


In [2]:
from datasets import load_dataset
ds_postpunk = load_dataset("AlekseyCalvin/song_lyrics_Ru2En_PostSoviet_alt_anthems", split="train")
print("Колонки:", ds_postpunk.column_names)

ds_postpunk = ds_postpunk.map(
    lambda x: {
        "ru": str(x["input"]).strip(),
        "en": str(x["accepted"]).strip()
    },
    remove_columns=["instruction", "input", "accepted", "rejected"]
)
ds_postpunk = ds_postpunk.filter(lambda x: len(x["en"]) > 0 and len(x["ru"]) > 0)
print(f"{len(ds_postpunk)} пар")

Колонки: ['instruction', 'input', 'accepted', 'rejected']
2030 пар


In [3]:
from datasets import load_dataset
ru_en_story_pairs = load_dataset("limloop/ru_en_story_pairs", split="train")
print("Колонки:", ru_en_story_pairs.column_names)

ru_en_story_pairs = ru_en_story_pairs.map(
    lambda x: {
        "ru": str(x["text_ru"]).strip(),
        "en": str(x["text_en"]).strip()
    },
    remove_columns=ru_en_story_pairs.column_names
)
ru_en_story_pairs = ru_en_story_pairs.filter(lambda x: len(x["en"]) > 0 and len(x["ru"]) > 0)
print(f"{len(ru_en_story_pairs)} пар")

Колонки: ['text_ru', 'text_en', 'summary_ru', 'summary_en', 'genre']
118499 пар


In [4]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
MAX_LENGTH = 128
def preprocess_function(examples):
    model_inputs = tokenizer(examples["en"], max_length=MAX_LENGTH, truncation=True)
    labels = tokenizer(examples["ru"], max_length=MAX_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [17]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
OUTPUT_DIR = "models/opus-mt-en-ru-postpunk"
ds_postpunk_split = ds_postpunk.train_test_split(test_size=0.15, seed=42)
tokenized_postpunk = ds_postpunk_split.map(preprocess_function, batched=True)
eval_dataset_postpunk = tokenized_postpunk["test"]
training_args_postpunk = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    num_train_epochs=3,
    predict_with_generate=False,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    warmup_ratio=0.1,
    report_to="none",
    dataloader_num_workers=2,
    dataloader_pin_memory=torch.cuda.is_available()
)

trainer_poetic = Seq2SeqTrainer(
    model=model,
    args=training_args_postpunk,
    train_dataset=tokenized_postpunk["train"],
    eval_dataset=eval_dataset_postpunk,
    data_collator=data_collator,
)

print(f"Запуск дообучения на postpunk. Train: {len(tokenized_postpunk['train'])}, Val: {len(eval_dataset_postpunk)}")
trainer_poetic.train()
trainer_poetic.save_model(OUTPUT_DIR)

Map: 100%|██████████| 305/305 [00:00<00:00, 6470.27 examples/s]


Запуск дообучения на postpunk. Train: 1725, Val: 305
{'loss': '6.311', 'grad_norm': '8.547', 'learning_rate': '2.32e-05', 'epoch': '0.9259'}
{'eval_loss': '1.654', 'eval_runtime': '12.96', 'eval_samples_per_second': '23.54', 'eval_steps_per_second': '1.544', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.43it/s]


{'loss': '3.482', 'grad_norm': '8.953', 'learning_rate': '1.289e-05', 'epoch': '1.852'}
{'eval_loss': '1.435', 'eval_runtime': '12.5', 'eval_samples_per_second': '24.39', 'eval_steps_per_second': '1.599', 'epoch': '2'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.65it/s]


{'loss': '3.092', 'grad_norm': '8.884', 'learning_rate': '2.577e-06', 'epoch': '2.778'}
{'eval_loss': '1.378', 'eval_runtime': '11.3', 'eval_samples_per_second': '26.99', 'eval_steps_per_second': '1.77', 'epoch': '3'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.70it/s]


{'train_runtime': '102.7', 'train_samples_per_second': '50.4', 'train_steps_per_second': '3.155', 'train_loss': '4.197', 'epoch': '3'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.59it/s]


In [5]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

OUTPUT_DIR = "models/opus-mt-en-ru_ru_en_story_pairs"

ru_en_story_pairs_split = ru_en_story_pairs.train_test_split(test_size=0.15, seed=42)

print("Токенизация датасета...")
tokenized_ru_en_story_pairs = ru_en_story_pairs_split.map(preprocess_function, batched=True)

train_dataset = tokenized_ru_en_story_pairs["train"]
eval_dataset = tokenized_ru_en_story_pairs["test"]
print("Тренировка модели...")
training_args_ru_en_story_pairs = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    num_train_epochs=3,
    predict_with_generate=False,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    warmup_ratio=0.1,
    report_to="none",
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    optim="adamw_torch",
)

trainer_ru_en = Seq2SeqTrainer(
    model=model,
    args=training_args_ru_en_story_pairs,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print(f"Запуск дообучения на ru_en_story_pairs. Train: {len(train_dataset)}, Val: {len(eval_dataset)}")
trainer_ru_en.train()
trainer_ru_en.save_model(OUTPUT_DIR)

Токенизация датасета...


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Тренировка модели...
Запуск дообучения на ru_en_story_pairs. Train: 100724, Val: 17775


Epoch,Training Loss,Validation Loss
1,0.724556,0.285193
2,0.516247,0.223303
3,0.470261,0.208043


C:\Users\magofrays\PycharmProjects\lb6\.venv\Lib\site-packages\transformers\integrations\sdpa_attention.py:92: UserWarning: Using AOTriton backend for Efficient Attention forward... (Triggered internally at C:/b/pytorch/aten/src/ATen/native/transformers/hip/attention.hip:1452.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
C:\Users\magofrays\PycharmProjects\lb6\.venv\Lib\site-packages\torch\autograd\graph.py:841: UserWarning: Using AOTriton backend for Efficient Attention backward... (Triggered internally at C:/b/pytorch/aten/src/ATen/native/transformers/hip/attention_backward.hip:551.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.52it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positio

In [6]:
import random

def get_test_samples(dataset, n=5, seed=42):
    random.seed(seed)
    if len(dataset) < n:
        raise ValueError(f"В датасете всего {len(dataset)} примеров, запрошено {n}")
    indices = random.sample(range(len(dataset)), n)
    return [dataset[i] for i in indices]

In [7]:
import evaluate

bleu_metric = evaluate.load("sacrebleu")

def compute_bleu(predictions, references):
    results = bleu_metric.compute(predictions=predictions, references=references)
    return results["score"]

In [12]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

MODEL_PATHS = {
    "Base": "Helsinki-NLP/opus-mt-en-ru",
    "Postpunk": "models/opus-mt-en-ru-postpunk",
    "Ru_en_story_pairs": "models/opus-mt-en-ru_ru_en_story_pairs",
}

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-ru")
bleu = evaluate.load("sacrebleu")

def translate_batch(texts, model, tokenizer, device):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, num_beams=4)
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)


loaded_models = {}
for name, path in MODEL_PATHS.items():
    try:
        model = AutoModelForSeq2SeqLM.from_pretrained(path).to(device)
        loaded_models[name] = model
        print(f"{name} загружена")
    except Exception as e:
        print(f"Ошибка загрузки {name}: {e}")

Loading weights: 100%|██████████| 258/258 [00:00<00:00, 85992.56it/s]


Base загружена


Loading weights: 100%|██████████| 254/254 [00:00<00:00, 15880.41it/s]


Postpunk загружена


Loading weights: 100%|██████████| 254/254 [00:00<00:00, 15875.20it/s]


Ru_en_story_pairs загружена


In [13]:
test_samples = get_test_samples(ds_postpunk, n=10, seed=41)

ens = [s["en"] for s in test_samples]
refs = [s["ru"] for s in test_samples]

In [14]:
if 'loaded_models' not in globals() or not loaded_models:
    raise RuntimeError("Модели не загружены")

results = {}

for name, model in loaded_models.items():
    preds = translate_batch(ens, model, tokenizer, device)
    score = compute_bleu(preds, refs)
    results[name] = score

base_score = results.get("Base", 0)
print("\n" + "="*65)
print(f"{'Модель':<35} | {'SacreBLEU':>10} | {'Diff':>10}")
print("-" * 65)
for name, score in results.items():
    diff = f"{score - base_score:+.2f}" if name != "Base" else "baseline"
    print(f"{name:<35} | {score:>10.2f} | {diff:>10}")
print("="*65)


Модель                              |  SacreBLEU |       Diff
-----------------------------------------------------------------
Base                                |       8.00 |   baseline
Postpunk                            |       3.14 |      -4.86
Ru_en_story_pairs                   |       3.48 |      -4.53


In [16]:
n = 0
print(f"EN:  {ens[n]}")
print(f"RU: {refs[n]}")
print('-'*60)

for name, model in loaded_models.items():
    pred = translate_batch([ens[n]], model, tokenizer, device)[0]
    print(f"{name}: {pred}")
    bleu = compute_bleu([pred], [[refs[n]]])
    print(f"{name} BLEU: {bleu:.2f}")
    print('-'*60)

EN:  Surf waves are dancing on, past the harbor's bend.
RU: Прямо за пристанью бьется волна.
------------------------------------------------------------
Base: Волны серфинга танцуют, мимо гавани.
Base BLEU: 6.57
------------------------------------------------------------
Postpunk: Воздат серфы по подобой гране.
Postpunk BLEU: 8.12
------------------------------------------------------------
Ru_en_story_pairs: Сурфовые волны танцуют, мимо изгиба гавани.
Ru_en_story_pairs BLEU: 5.52
------------------------------------------------------------


In [10]:
trainer_ru_en.save_model(OUTPUT_DIR)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.77it/s]
